#### Gurobi Demo R01

A concise MILP demo using a capital budgeting / knapsack model

- builds and solves MILP
- solves the LP relaxation
- compares objective values and selected projects
- prints diagnostics

Solved issues: 

- figured out gurobi license
- activation & environment issues. 
- recreated toy buses using gurobi
- built better MILP demo with gurobi (knapsack demo)
- compared LP to MILP
- gurobi diagnostics



In [8]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import time

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

In [9]:
# Data: cost and value of candidate projects
projects = pd.DataFrame({
    'Project': ['P1','P2','P3','P4','P5','P6','P7','P8'],
    'Cost':    [12,   7,   11,   8,   9,   6,   10,   5],
    'Value':   [24,  13,   27,  17,  16,  10,   19,   9]
})

budget = 30
projects

,Project,Cost,Value
0,P1,12,24
1,P2,7,13
2,P3,11,27
3,P4,8,17
4,P5,9,16
5,P6,6,10
6,P7,10,19
7,P8,5,9


#### Model notes

Decision variable:  
- $x_j \in \{0,1\}$ = select project $j$

Objective:  
- maximize total value

Constraints:
- budget cap
- select **at most 4** projects
- **P1 and P3** are mutually exclusive
- if **P7** is selected, then **P2** must also be selected
- select **at least one** of **P4 or P8**


In [10]:
def build_model(binary=True):
    m = gp.Model('capital_budgeting_demo')
    m.Params.OutputFlag = 1

    vtype = GRB.BINARY if binary else GRB.CONTINUOUS
    x = m.addVars(projects.index, vtype=vtype, lb=0, ub=1, name='x')

    # Objective
    m.setObjective(gp.quicksum(projects.loc[i, 'Value'] * x[i] for i in projects.index), GRB.MAXIMIZE)

    # Constraints
    m.addConstr(gp.quicksum(projects.loc[i, 'Cost'] * x[i] for i in projects.index) <= budget, name='budget')
    m.addConstr(gp.quicksum(x[i] for i in projects.index) <= 4, name='max_projects')
    m.addConstr(x[0] + x[2] <= 1, name='mutually_exclusive_P1_P3')   # P1 and P3 cannot both be chosen
    m.addConstr(x[6] <= x[1], name='P7_implies_P2')                  # if P7 then P2
    m.addConstr(x[3] + x[7] >= 1, name='choose_P4_or_P8')            # at least one of P4 or P8

    return m, x

In [11]:
# Solve LP relaxation first
lp, x_lp = build_model(binary=False)
t0 = time.time()
lp.optimize()
lp_wall = time.time() - t0

lp_sol = pd.DataFrame({
    'Project': projects['Project'],
    'x_LP': [x_lp[i].X for i in projects.index]
})
print('\nLP relaxation objective:', round(lp.ObjVal, 4))
lp_sol

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Optimize a model with 5 rows, 8 columns and 22 nonzeros
Model fingerprint: 0x7285e5c2
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [9e+00, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+01]
Presolve time: 0.02s
Presolved: 5 rows, 8 columns, 22 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.3636364e+01   7.909091e+00   0.000000e+00      0s
       3    6.4705882e+01   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.03 seconds (0.00 work units)
Optimal objective  6.470588235e+01

LP relaxation 

,Project,x_LP
0,P1,0.000
1,P2,0.647
2,P3,1.000
3,P4,1.000
4,P5,0.000
5,P6,0.000
6,P7,0.647
7,P8,0.000


In [12]:
# Solve MILP
mip, x_mip = build_model(binary=True)
t0 = time.time()
mip.optimize()
mip_wall = time.time() - t0

mip_sol = pd.DataFrame({
    'Project': projects['Project'],
    'Selected': [int(round(x_mip[i].X)) for i in projects.index],
    'Cost': projects['Cost'],
    'Value': projects['Value']
})

chosen = mip_sol[mip_sol['Selected'] == 1].copy()
chosen

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Optimize a model with 5 rows, 8 columns and 22 nonzeros
Model fingerprint: 0x6ae7e236
Variable types: 0 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [9e+00, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+01]
Found heuristic solution: objective 56.0000000
Presolve removed 1 rows and 1 columns
Presolve time: 0.01s
Presolved: 4 rows, 7 columns, 18 nonzeros
Found heuristic solution: objective 32.0000000
Variable types: 0 continuous, 7 integer (7 binary)
Found heuristic solution: objective 59.0000000

Root relaxation: obj

,Project,Selected,Cost,Value
2,P3,1,11,27
3,P4,1,8,17
5,P6,1,6,10
7,P8,1,5,9


In [13]:
print('=== Gurobi Diagnostics ===')
print(f'Status         : {mip.Status}')
print(f'MILP objective : {mip.ObjVal:.4f}')
print(f'Best bound     : {mip.ObjBound:.4f}')
print(f'MIP gap        : {mip.MIPGap:.6f}')
print(f'Node count     : {mip.NodeCount}')
print(f'Runtime (s)    : {mip.Runtime:.4f}')
print(f'Wall time (s)  : {mip_wall:.4f}')

lp_obj = lp.ObjVal
mip_obj = mip.ObjVal
gap_pct = 100 * (lp_obj - mip_obj) / lp_obj if lp_obj != 0 else 0

print('\n=== Model Summary ===')
print(f'LP relaxation objective : {lp_obj:.4f}')
print(f'MILP objective          : {mip_obj:.4f}')
print(f'LP -> MILP gap (%)      : {gap_pct:.2f}')
print(f'Total cost used         : {chosen["Cost"].sum()} / {budget}')
print(f'Projects selected       : {list(chosen["Project"])}')

=== Gurobi Diagnostics ===
Status         : 2
MILP objective : 63.0000
Best bound     : 63.0000
MIP gap        : 0.000000
Node count     : 1.0
Runtime (s)    : 0.0500
Wall time (s)  : 0.0529

=== Model Summary ===
LP relaxation objective : 64.7059
MILP objective          : 63.0000
LP -> MILP gap (%)      : 2.64
Total cost used         : 30 / 30
Projects selected       : ['P3', 'P4', 'P6', 'P8']


In [14]:
# Side-by-side comparison
summary = projects[['Project', 'Cost', 'Value']].copy()
summary['x_LP'] = [x_lp[i].X for i in projects.index]
summary['Selected_MILP'] = [int(round(x_mip[i].X)) for i in projects.index]
summary

,Project,Cost,Value,x_LP,Selected_MILP
0,P1,12,24,0.000,0
1,P2,7,13,0.647,0
2,P3,11,27,1.000,1
3,P4,8,17,1.000,1
4,P5,9,16,0.000,0
5,P6,6,10,0.000,1
6,P7,10,19,0.647,0
7,P8,5,9,0.000,1


#### Notes

- The **LP relaxation** may pick fractional projects.
- The **MILP** enforces true yes/no decisions.
- The difference between LP and MILP objective is a simple, useful illustration of the **integrality gap**.
- NodeCount, Best bound, and MIPGap are the main Gurobi outputs to watch when the model gets larger or harder.
